In [ ]:
import xarray as xr

In [ ]:
expert_003_ds = xr.open_zarr("./data/SWOT_L3_Expert_v3_0_pass003_cg.zarr")
expert_016_ds = xr.open_zarr("./data/SWOT_L3_Expert_v3_0_pass016_cg.zarr")

In [ ]:
expert_003_ds = expert_003_ds.reset_coords(["time", "latitude", "longitude"])
expert_016_ds = expert_016_ds.reset_coords(["time", "latitude", "longitude"])

In [ ]:
min_longitude = min(ds.longitude.min().values for ds in [expert_003_ds, expert_016_ds])
max_longitude = max(ds.longitude.max().values for ds in [expert_003_ds, expert_016_ds])
min_latitude = min(ds.latitude.min().values for ds in [expert_003_ds, expert_016_ds])
max_latitude = max(ds.latitude.max().values for ds in [expert_003_ds, expert_016_ds])

In [ ]:
expert_003_ds = expert_003_ds.expand_dims("pass_num").assign_coords(pass_num=["003"])
expert_016_ds = expert_016_ds.expand_dims("pass_num").assign_coords(pass_num=["016"])

In [ ]:
expert_ds = xr.concat([expert_003_ds, expert_016_ds], dim="pass_num", join="outer")

In [ ]:
expert_ds = expert_ds.chunk({"num_lines": -1, "num_pixels": -1})

expert_ds["latitude"] = expert_ds.latitude.interpolate_na(dim="num_pixels", method="linear", fill_value="extrapolate")
expert_ds["latitude"] = expert_ds.latitude.interpolate_na(dim="num_lines", method="linear", fill_value="extrapolate")

expert_ds["longitude"] = expert_ds.longitude.interpolate_na(dim="num_pixels", method="linear", fill_value="extrapolate")
expert_ds["longitude"] = expert_ds.longitude.interpolate_na(dim="num_lines", method="linear", fill_value="extrapolate")

expert_ds = expert_ds.set_coords(["latitude", "longitude"])

In [ ]:
for var in expert_ds.data_vars:
    if "chunks" in expert_ds[var].encoding:
        del expert_ds[var].encoding["chunks"]
expert_ds = expert_ds.chunk({"cycle_num": 10, "num_lines": -1, "num_pixels": -1})

In [ ]:
expert_ds.attrs["min_longitude"] = min_longitude
expert_ds.attrs["max_longitude"] = max_longitude
expert_ds.attrs["min_latitude"] = min_latitude
expert_ds.attrs["max_latitude"] = max_latitude

In [ ]:
expert_ds.to_zarr("./data/SWOT_L3_Expert_v3_0_pass003_pass016_cg.zarr", mode="w")